# Topic: Hash Generator and Verifier - One-way hashing, chunked file hashing, and salted hash verification (PBKDF2)
 
## 1. CRYPTOGRAPHIC HASH FUNCTIONS
- **Cryptographic Hash**: A one-way mathematical function that maps input data of arbitrary size to a fixed-size bit representation (message digest).
- **Properties**:
  1. **One-way (Pre-image resistance)**: Computationally infeasible to reverse engineer the input from its output digest.
  2. **Collision Resistant**: Infeasible to find two distinct inputs that produce the same output digest.
  3. **Avalanche Effect**: A tiny change in input (e.g., swapping one bit) completely changes the output digest.
 
## 2. VULNERABILITIES & MITIGATIONS
- **Rainbow Table Attack**: Attackers use precomputed lists of plaintexts and their corresponding hashes to reverse lookup passwords.
- **Mitigation (Salting)**:
  - Append a unique, cryptographically secure random string (Salt) to the password before hashing:
    $$\text{Hash} = \text{HashFunction}(\text{Password} + \text{Salt})$$
  - This guarantees that two identical passwords generate different digests, rendering rainbow tables useless.
 
---


In [ ]:
import hashlib
import os

# 1. Demonstrating standard hashlib hashing
print("--- Hash Generation (hashlib) ---")
message = b"Secret data payload"

# Generate SHA-256 hash
sha256_hash = hashlib.sha256(message).hexdigest()
print(f"Message: {message.decode()}")
print(f"SHA-256: {sha256_hash}")



In [ ]:
# 2. Chunk-based File Hashing (Memory-Efficient Best Practice)
# Instead of loading a large file into RAM at once, read it in chunks (e.g., 4096 bytes).
def calculate_file_sha256(file_path, chunk_size=4096):
    """Computes SHA-256 of a file incrementally (O(1) space)."""
    hasher = hashlib.sha256()
    with open(file_path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            hasher.update(chunk)
    return hasher.hexdigest()

temp_file = "hash_file.txt"
with open(temp_file, "w") as f:
    f.write("A large text file contents to verify integrity.")

file_hash = calculate_file_sha256(temp_file)
print(f"\nFile Hash: {file_hash}")

if os.path.exists(temp_file):
    os.remove(temp_file)



In [ ]:
# 3. Salted Password Hashing using PBKDF2 (Password-Based Key Derivation Function 2)
print("\n--- Salted Password Hashing (PBKDF2) ---")
password = "UserSecretPassword123"

# Generate random 16-byte salt
salt = os.urandom(16)

# Derive hash using PBKDF2 with HMAC-SHA256 and 100,000 iterations
derived_hash = hashlib.pbkdf2_hmac(
    hash_name='sha256',
    password=password.encode('utf-8'),
    salt=salt,
    iterations=100000
)

# Represent salt and hash in hexadecimal for storage
print(f"Password:   {password}")
print(f"Salt (hex): {salt.hex()}")
print(f"Hash (hex): {derived_hash.hex()}")

# Verification Routine
def verify_password(stored_salt, stored_hash, input_password):
    new_hash = hashlib.pbkdf2_hmac(
        hash_name='sha256',
        password=input_password.encode('utf-8'),
        salt=stored_salt,
        iterations=100000
    )
    return new_hash == stored_hash

print(f"\nVerification (Correct Password): {verify_password(salt, derived_hash, password)}")  # Expected: True
print(f"Verification (Incorrect Password): {verify_password(salt, derived_hash, 'WrongPass')}")  # Expected: False
